In [2]:
from dotenv import load_dotenv
import os
import pandas as pd
from faker import Faker
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.syntax import Syntax
from supabase import create_client, Client
from openai import OpenAI
import random
from datetime import datetime, timedelta

load_dotenv()

console = Console()
fake = Faker()

# Validate env vars
required = ["SUPABASE_URL", "SUPABASE_KEY", "OPENAI_API"]
missing = [v for v in required if not os.getenv(v)]
if missing:
    raise EnvironmentError(f"Missing: {missing}")

# Initialize Supabase client
supabase: Client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

# Initialize Gemini AI client (OpenAI-compatible)
ai_client = OpenAI(
    api_key=os.getenv("OPENAI_API")
)

console.print("[bold green]✅ All clients initialized successfully![/bold green]")


✅ All clients initialized successfully!

In [3]:
# Database schema context (used by AI to generate accurate SQL)
DB_SCHEMA = """
Table: customers
  - id SERIAL PRIMARY KEY
  - name TEXT NOT NULL
  - email TEXT UNIQUE NOT NULL
  - phone TEXT
  - city TEXT
  - country TEXT
  - created_at TIMESTAMPTZ

Table: products
  - id SERIAL PRIMARY KEY
  - name TEXT NOT NULL
  - category TEXT NOT NULL (Electronics, Clothing, Books, Home and Garden, Sports, Food and Beverages)
  - price NUMERIC(10,2) NOT NULL
  - stock_quantity INTEGER
  - created_at TIMESTAMPTZ

Table: orders
  - id SERIAL PRIMARY KEY
  - customer_id INTEGER REFERENCES customers(id)
  - order_date TIMESTAMPTZ
  - status TEXT -- one of: pending, processing, shipped, delivered, cancelled
  - total_amount NUMERIC(10,2)

Table: order_items
  - id SERIAL PRIMARY KEY
  - order_id INTEGER REFERENCES orders(id)
  - product_id INTEGER REFERENCES products(id)
  - quantity INTEGER NOT NULL
  - unit_price NUMERIC(10,2) NOT NULL

Table: reviews
  - id SERIAL PRIMARY KEY
  - product_id INTEGER REFERENCES products(id)
  - customer_id INTEGER REFERENCES customers(id)
  - rating INTEGER (1 to 5)
  - comment TEXT
  - created_at TIMESTAMPTZ
"""

console.print(Panel(DB_SCHEMA, title="📊 Database Schema", border_style="cyan"))


╭────────────────────────────────────────────── 📊 Database Schema ───────────────────────────────────────────────╮
│                                                                                                                 │
│ Table: customers                                                                                                │
│   - id SERIAL PRIMARY KEY                                                                                       │
│   - name TEXT NOT NULL                                                                                          │
│   - email TEXT UNIQUE NOT NULL                                                                                  │
│   - phone TEXT                                                                                                  │
│   - city TEXT                                                                                                   │
│   - country TEXT                                                                                                │
│   - created_at TIMESTAMPTZ                                                                                      │
│                                                                                                                 │
│ Table: products                                                                                                 │
│   - id SERIAL PRIMARY KEY                                                                                       │
│   - name TEXT NOT NULL                                                                                          │
│   - category TEXT NOT NULL (Electronics, Clothing, Books, Home and Garden, Sports, Food and Beverages)          │
│   - price NUMERIC(10,2) NOT NULL                                                                                │
│   - stock_quantity INTEGER                                                                                      │
│   - created_at TIMESTAMPTZ                                                                                      │
│                                                                                                                 │
│ Table: orders                                                                                                   │
│   - id SERIAL PRIMARY KEY                                                                                       │
│   - customer_id INTEGER REFERENCES customers(id)                                                                │
│   - order_date TIMESTAMPTZ                                                                                      │
│   - status TEXT -- one of: pending, processing, shipped, delivered, cancelled                                   │
│   - total_amount NUMERIC(10,2)                                                                                  │
│                                                                                                                 │
│ Table: order_items                                                                                              │
│   - id SERIAL PRIMARY KEY                                                                                       │
│   - order_id INTEGER REFERENCES orders(id)                                                                      │
│   - product_id INTEGER REFERENCES products(id)                                                                  │
│   - quantity INTEGER NOT NULL                                                                                   │
│   - unit_price NUMERIC(10,2) NOT NULL                                                                           │
│                                                                                                                 │
│ Table: reviews                                                                                                  │
│   - id SERIAL PRIMARY KEY                              

## 📦 Generate and Insert Fake Data
Uses Faker to create realistic e-commerce data and inserts it into Supabase.

In [4]:
# == Generate Fake Customers ==
customers_data = []
for i in range(50):
    customers_data.append({
        "name": fake.name(),
        "email": fake.unique.email(),
        "phone": fake.phone_number(),
        "city": fake.city(),
        "country": fake.country(),
    })

res = supabase.table("customers").insert(customers_data).execute()
console.print(f"[green]✅ Inserted {len(res.data)} customers[/green]")

# == Generate Fake Products ==
categories = ["Electronics", "Clothing", "Books", "Home and Garden", "Sports", "Food and Beverages"]
products_data = []
for i in range(30):
    products_data.append({
        "name": fake.catch_phrase(),
        "category": random.choice(categories),
        "price": round(random.uniform(5.99, 999.99), 2),
        "stock_quantity": random.randint(0, 500),
    })

res = supabase.table("products").insert(products_data).execute()
console.print(f"[green]✅ Inserted {len(res.data)} products[/green]")

# == Fetch IDs for foreign keys ==
customer_ids = [c["id"] for c in supabase.table("customers").select("id").execute().data]
product_ids = [p["id"] for p in supabase.table("products").select("id").execute().data]
product_prices = {p["id"]: float(p["price"]) for p in supabase.table("products").select("id, price").execute().data}

# == Generate Fake Orders with Order Items ==
statuses = ["pending", "processing", "shipped", "delivered", "cancelled"]

for _ in range(80):
    cid = random.choice(customer_ids)
    order_date = fake.date_time_between(start_date="-90d", end_date="now").isoformat()
    status = random.choice(statuses)

    # Create the order first
    order_res = supabase.table("orders").insert({
        "customer_id": cid,
        "order_date": order_date,
        "status": status,
        "total_amount": 0
    }).execute()
    order_id = order_res.data[0]["id"]

    # Add 1-5 items per order
    num_items = random.randint(1, 5)
    total = 0.0
    items = []
    for _ in range(num_items):
        pid = random.choice(product_ids)
        qty = random.randint(1, 4)
        price = product_prices[pid]
        total += price * qty
        items.append({
            "order_id": order_id,
            "product_id": pid,
            "quantity": qty,
            "unit_price": price
        })

    supabase.table("order_items").insert(items).execute()
    supabase.table("orders").update({"total_amount": round(total, 2)}).eq("id", order_id).execute()

console.print(f"[green]✅ Inserted 80 orders with items[/green]")

# == Generate Fake Reviews ==
reviews_data = []
for _ in range(100):
    reviews_data.append({
        "product_id": random.choice(product_ids),
        "customer_id": random.choice(customer_ids),
        "rating": random.randint(1, 5),
        "comment": fake.sentence(nb_words=12),
    })

res = supabase.table("reviews").insert(reviews_data).execute()
console.print(f"[green]✅ Inserted {len(res.data)} reviews[/green]")
console.print("[bold cyan]\n🎉 All fake data generated and inserted![/bold cyan]")


✅ Inserted 50 customers

✅ Inserted 30 products

✅ Inserted 80 orders with items

✅ Inserted 100 reviews

🎉 All fake data generated and inserted!

In [5]:
# == Verify: Show row counts and sample data ==
tables = ["customers", "products", "orders", "order_items", "reviews"]

summary = Table(title="📊 Database Summary", show_lines=True)
summary.add_column("Table", style="cyan bold")
summary.add_column("Row Count", style="green", justify="right")

for t in tables:
    data = supabase.table(t).select("id", count="exact").execute()
    summary.add_row(t, str(data.count))

console.print(summary)

# Show sample customers
sample = supabase.table("customers").select("*").limit(5).execute()
df = pd.DataFrame(sample.data)
console.print("\n[bold]Sample Customers:[/bold]")
console.print(df.to_string(index=False))


    📊 Database Summary    
┏━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Table       ┃ Row Count ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ customers   │       300 │
├─────────────┼───────────┤
│ products    │       180 │
├─────────────┼───────────┤
│ orders      │       480 │
├─────────────┼───────────┤
│ order_items │      1432 │
├─────────────┼───────────┤
│ reviews     │       600 │
└─────────────┴───────────┘

Sample Customers:

id              name                     email                 phone            city                     country  
created_at
  1     Randy Elliott     michael38@example.org     (647)866-5249x878 Port Jeremyfurt French Southern Territories 
2026-06-29T10:36:35.441369+00:00
  2     David Goodman       emily23@example.net    (529)596-1968x2716   Nicholashaven                       Egypt 
2026-06-29T10:36:35.441369+00:00
  3   Sarah Jefferson        vwhite@example.org 001-375-295-4266x1628   Jenniferburgh                  Bangladesh 
2026-06-29T10:36:35.441369+00:00
  4      Nicole Riley   snyderscott@example.org          710.906.9479    North Robert                  Luxembourg 
2026-06-29T10:36:35.441369+00:00
  5 Brittany Espinoza kellyjohnston@example.org         (982)296-4691    Martinezside                    Barbados 
2026-06-29T10:36:35.441369+00:00

## 🤖 AI SQL Engine
Core functions: convert natural language to SQL, execute it, and display results beautifully.

In [9]:
SYSTEM_PROMPT = f"""You are an expert PostgreSQL SQL assistant.
Given the following database schema, generate ONLY a valid PostgreSQL SQL query.
Do NOT include any explanation, markdown formatting, or code fences.
Return ONLY the raw SQL query.

DATABASE SCHEMA:
{DB_SCHEMA}

RULES:
- Use only tables and columns from the schema above
- Always use proper JOINs when combining tables
- Use aggregate functions like COUNT, SUM, AVG when appropriate
- Return readable column aliases
- Limit results to 20 rows unless the user specifies otherwise
- For date filtering use relative dates like CURRENT_DATE - INTERVAL
"""


def nl_to_sql(question: str) -> str:
    """Convert a natural language question to SQL using AI."""
    response = ai_client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ],
        temperature=0.1,
    )
    sql = response.choices[0].message.content.strip()
    # Clean up any accidental markdown fences
    sql = sql.replace("```sql", "").replace("```", "").strip()
    # Strip trailing semicolons (Supabase RPC does not accept them)
    sql = sql.rstrip(";").strip()
    return sql


def run_query(sql: str) -> list:
    """Execute SQL via Supabase RPC and return results."""
    result = supabase.rpc("execute_sql", {"query": sql}).execute()
    return result.data if result.data else []


def display_results(data: list, title: str = "Query Results"):
    """Display query results as a Rich table."""
    if not data:
        console.print("[yellow]No results found.[/yellow]")
        return

    table = Table(title=title, show_lines=True, border_style="bright_blue")

    # Add columns from first row keys
    for col in data[0].keys():
        table.add_column(str(col), style="cyan")

    # Add rows
    for row in data:
        table.add_row(*[str(v) for v in row.values()])

    console.print(table)
    console.print(f"[dim]{len(data)} row(s) returned[/dim]")


def ask(question: str):
    """Main function: ask a question in natural language, get SQL + results."""
    console.print(Panel(question, title="Question", border_style="yellow"))

    # Generate SQL
    sql = nl_to_sql(question)
    console.print(Panel(Syntax(sql, "sql", theme="monokai", line_numbers=True),
                        title="Generated SQL", border_style="green"))

    # Execute
    try:
        results = run_query(sql)
        display_results(results)
    except Exception as e:
        console.print(f"[bold red]Error executing SQL: {e}[/bold red]")


console.print("[bold green]✅ AI SQL Engine ready! Use ask() to query.[/bold green]")


✅ AI SQL Engine ready! Use ask() to query.

In [10]:
ask("Show me the top 5 customers who spent the most money")


╭─────────────────────────────────────────────────── Question ────────────────────────────────────────────────────╮
│ Show me the top 5 customers who spent the most money                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Generated SQL ─────────────────────────────────────────────────╮
│   1 SELECT c.id AS customer_id, c.name AS customer_name, c.email AS customer_email, SUM(o.total_amount) AS tota │
│   2 FROM customers c                                                                                            │
│   3 JOIN orders o ON c.id = o.customer_id                                                                       │
│   4 GROUP BY c.id, c.name, c.email                                                                              │
│   5 ORDER BY total_spent DESC                                                                                   │
│   6 LIMIT 5                                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                              Query Results                               
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ customer_id ┃ customer_name ┃ customer_email             ┃ total_spent ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ 39          │ Adam Wall     │ abigail14@example.org      │ 33661.51    │
├─────────────┼───────────────┼────────────────────────────┼─────────────┤
│ 19          │ Eddie Paul    │ sarah97@example.net        │ 28735.85    │
├─────────────┼───────────────┼────────────────────────────┼─────────────┤
│ 21          │ Chris Payne   │ larryjackson@example.org   │ 26084.35    │
├─────────────┼───────────────┼────────────────────────────┼─────────────┤
│ 1           │ Randy Elliott │ michael38@example.org      │ 25013.31    │
├─────────────┼───────────────┼────────────────────────────┼─────────────┤
│ 33          │ Amy Rosario   │ masonelizabeth@example.net │ 24896.07    │
└─────────────┴───────────────┴────────────────────────────┴─────────────┘

5 row(s) returned

In [ ]:
ask("What is the average rating for each product category?")


In [ ]:
ask("List all products that have never been ordered")


## Interactive AI SQL Editor
Run the cell below to start an interactive loop. Type your questions and get instant SQL + results.
Type `quit` or `exit` to stop.

In [11]:
# == Interactive AI SQL Editor (Answer Only Mode) ==

def ask_answer(question: str):
    """Ask a question in natural language and show ONLY the results (no SQL)."""
    console.print(Panel(question, title="Question", border_style="yellow"))

    # Generate SQL (hidden from user)
    sql = nl_to_sql(question)

    # Execute and show only the results
    try:
        results = run_query(sql)
        display_results(results, title="Answer")
    except Exception as e:
        console.print(f"[bold red]Error: {e}[/bold red]")


console.print(Panel(
    "[bold]Type a question in plain English to get the answer directly.\n"
    "Type [red]quit[/red] or [red]exit[/red] to stop.[/bold]",
    title="AI SQL Editor (Answer Mode)",
    border_style="bright_magenta"
))

while True:
    console.print()
    question = input("Ask a question: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        console.print("[bold cyan]Goodbye![/bold cyan]")
        break
    if not question:
        continue
    ask_answer(question)


╭────────────────────────────────────────── AI SQL Editor (Answer Mode) ──────────────────────────────────────────╮
│ Type a question in plain English to get the answer directly.                                                    │
│ Type quit or exit to stop.                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Question ────────────────────────────────────────────────────╮
│ List all products that have never been ordered                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                      Answer                                                       
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ id  ┃ name                         ┃ category           ┃ price  ┃ stock_quantity ┃ created_at                  ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 133 │ Fundamental user-facing      │ Food and Beverages │ 673.83 │ 233            │ 2026-06-30T11:26:26.607521… │
│     │ alliance                     │                    │        │                │                             │
├─────┼──────────────────────────────┼────────────────────┼────────┼────────────────┼─────────────────────────────┤
│ 153 │ Public-key next generation   │ Sports             │ 965.93 │ 43             │ 2026-06-30T11:43:50.018905… │
│     │ capability                   │                    │        │                │                             │
├─────┼──────────────────────────────┼────────────────────┼────────┼────────────────┼─────────────────────────────┤
│ 156 │ Synergistic scalable         │ Sports             │ 948.92 │ 101            │ 2026-06-30T11:43:50.018905… │
│     │ intranet                     │                    │        │                │                             │
├─────┼──────────────────────────────┼────────────────────┼────────┼────────────────┼─────────────────────────────┤
│ 162 │ Phased full-range pricing    │ Home and Garden    │ 889.04 │ 50             │ 2026-06-30T11:43:50.018905… │
│     │ structure                    │                    │        │                │                             │
├─────┼──────────────────────────────┼────────────────────┼────────┼────────────────┼─────────────────────────────┤
│ 164 │ Front-line multi-state       │ Clothing           │ 825.09 │ 359            │ 2026-06-30T11:43:50.018905… │
│     │ contingency                  │                    │        │                │                             │
├─────┼──────────────────────────────┼────────────────────┼────────┼────────────────┼─────────────────────────────┤
│ 170 │ Innovative neutral           │ Books              │ 256.13 │ 333            │ 2026-06-30T11:43:50.018905… │
│     │ collaboration                │                    │        │                │                             │
├─────┼──────────────────────────────┼────────────────────┼────────┼────────────────┼─────────────────────────────┤
│ 172 │ Innovative actuating         │ Sports             │ 163.69 │ 156            │ 2026-06-30T11:43:50.018905… │
│     │ throughput                   │                    │        │                │                             │
├─────┼──────────────────────────────┼────────────────────┼────────┼────────────────┼─────────────────────────────┤
│ 174 │ Compatible maximized         │ Food and Beverages │ 329.67 │ 111            │ 2026-06-30T11:43:50.018905… │
│     │ strategy                     │                    │        │                │                             │
├─────┼──────────────────────────────┼────────────────────┼────────┼────────────────┼─────────────────────────────┤
│ 178 │ Optimized zero               │ Food and Beverages │ 196.04 │ 192            │ 2026-06-30T11:43:50.018905… │
│     │ administration Local Area    │                    │        │                │                             │
│     │ Network                      │                    │        │                │                             │
└─────┴──────────────────────────────┴────────────────────┴────────┴────────────────┴─────────────────────────────┘

9 row(s) returned

Goodbye!